In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

df = pd.read_parquet("../../data/processed/pickup_features_v2.parquet")
clean = pd.read_parquet("../../data/processed/pickup_clean.parquet",
                        columns=['order_id','courier_id','aoi_id'])
df = df.merge(clean, on='order_id', how='left')
print(f"Loaded {len(df):,} rows")

df = df.sort_values('ds')

courier_features = df.groupby('courier_id').agg(
    courier_late_rate     =('courier_late_rate', 'last'),
    courier_running_rate  =('courier_running_rate', 'last'),
    courier_city_rate     =('courier_city_rate', 'last'),
    courier_zone_familiarity=('courier_zone_familiarity', 'last'),
    courier_tenure_days   =('courier_tenure_days', 'last'),
    courier_load_3h       =('courier_load_3h', 'last'),
    courier_orders_so_far =('courier_orders_so_far', 'last'),
    mins_since_last_accept=('mins_since_last_accept', 'last'),
    orders_per_courier    =('orders_per_courier', 'last'),
    velocity_target       =('velocity_target', 'last'),
    zone_running_rate     =('zone_running_rate', 'last'),
).to_dict('index')

aoi_features = df.groupby('aoi_id')['aoi_disruption_rate'].last().to_dict()

df['city_hour_key'] = df['city'].astype(str) + '_' + df['hour_of_day'].astype(str)
city_hour_features = df.groupby('city_hour_key')['city_hour_rate'].last().to_dict()

GLOBAL = {
    'courier_late_rate': df['courier_late_rate'].mean(),
    'aoi_disruption_rate': df['aoi_disruption_rate'].mean(),
    'city_hour_rate': df['city_hour_rate'].mean(),
    'courier_running_rate': df['courier_running_rate'].mean(),
    'courier_city_rate': df['courier_city_rate'].mean(),
    'zone_running_rate': df['zone_running_rate'].mean(),
}

import joblib
store = {'courier': courier_features, 'aoi': aoi_features,
         'city_hour': city_hour_features, 'global': GLOBAL}
joblib.dump(store, "../../models/feature_store.pkl")
print(f"Feature store saved:")
print(f"  {len(courier_features):,} couriers")
print(f"  {len(aoi_features):,} zones")
print(f"  {len(city_hour_features):,} city-hour combos")

Loaded 6,064,908 rows
Feature store saved:
  15,928 couriers
  24,491 zones
  97 city-hour combos


In [ ]:
model = lgb.Booster(model_file="../../models/pickup_disruption_lgbm.txt")

MODEL_FEATURES = ['hour_of_day','courier_orders_so_far','day_of_week','accept_distance_km',
                  'courier_late_rate','aoi_disruption_rate','city_hour_rate','courier_city_rate',
                  'courier_zone_familiarity','courier_tenure_days','courier_load_3h',
                  'mins_since_last_accept','velocity_target','courier_running_rate',
                  'zone_running_rate','orders_per_courier','city']
test = df[df['ds'] >= 951].copy()
X = test[MODEL_FEATURES].copy()
for c in ['city','day_of_week','hour_of_day']:
    X[c] = X[c].astype('category')

scores = model.predict(X)
print("Predicted risk score distribution (test set):")
print(pd.Series(scores).describe(percentiles=[.5,.75,.90,.95,.99]).to_string())

p90, p95 = np.percentile(scores, [90, 95])
print(f"\nProposed thresholds (from actual score distribution):")
print(f"  high   (top 5%):  score >= {p95:.4f}")
print(f"  medium (top 10%): score >= {p90:.4f}")
print(f"  low:              below {p90:.4f}")

Predicted risk score distribution (test set):
count    1.118595e+06
mean     2.649530e-01
std      2.474012e-01
min      2.141016e-04
50%      1.972922e-01
75%      4.196450e-01
90%      6.459361e-01
95%      7.677224e-01
99%      9.329720e-01
max      9.932981e-01

Proposed thresholds (from actual score distribution):
  high   (top 5%):  score >= 0.7677
  medium (top 10%): score >= 0.6459
  low:              below 0.6459


In [ ]:
test = df[df['ds'] >= 951].copy()
X = test[MODEL_FEATURES].copy()
for c in ['city','day_of_week','hour_of_day']:
    X[c] = X[c].astype('category')
test['score'] = model.predict(X)

high_ex   = test[test['score'] >= 0.768].iloc[0]
medium_ex = test[(test['score'] >= 0.646) & (test['score'] < 0.768)].iloc[0]
low_ex    = test[test['score'] < 0.646].sort_values('score').iloc[0]  # a clearly low one

for label, ex in [('HIGH', high_ex), ('MEDIUM', medium_ex), ('LOW', low_ex)]:
    print(f"\n=== {label} (score={ex['score']:.4f}) ===")
    print(f'  "courier_id": {int(ex["courier_id"])},')
    print(f'  "city": "{ex["city"]}",')
    print(f'  "aoi_id": {int(ex["aoi_id"])},')
    print(f'  "hour_of_day": {int(ex["hour_of_day"])},')
    print(f'  "day_of_week": {int(ex["day_of_week"])},')
    print(f'  "accept_distance_km": {ex["accept_distance_km"]:.2f}')


=== HIGH (score=0.9704) ===
  "courier_id": 698,
  "city": "Shanghai",
  "aoi_id": 19320,
  "hour_of_day": 8,
  "day_of_week": 5,
  "accept_distance_km": nan

=== MEDIUM (score=0.7217) ===
  "courier_id": 8306,
  "city": "Chongqing",
  "aoi_id": 2788,
  "hour_of_day": 7,
  "day_of_week": 5,
  "accept_distance_km": nan

=== LOW (score=0.0002) ===
  "courier_id": 205,
  "city": "Yantai",
  "aoi_id": 4117,
  "hour_of_day": 15,
  "day_of_week": 2,
  "accept_distance_km": 0.26
